# UNESCO World Heritage

## 0 — Automatically download and prepare UNESCO data

In [1]:
import pandas as pd
import numpy as np
import requests
import plotly.express as px
from io import BytesIO
from IPython.display import display

SOURCE_URL = "https://data.unesco.org/api/explore/v2.1/catalog/datasets/whc001/exports/csv?lang=en&timezone=UTC"

response = requests.get(SOURCE_URL, timeout=90, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()
raw = pd.read_csv(BytesIO(response.content), sep=";", low_memory=False)
# Defensive fallback for an unexpected delimiter.
if len(raw.columns) == 1:
    raw = pd.read_csv(BytesIO(response.content), low_memory=False)

required = ["name_en", "states_names", "category", "area_hectares", "short_description_en"]
missing = [c for c in required if c not in raw.columns]
if missing:
    raise ValueError(f"Unexpected UNESCO export schema. Missing: {missing}. Available: {raw.columns.tolist()}")

df = raw.rename(columns={
    "name_en": "Name", "states_names": "Country", "category": "Category",
    "area_hectares": "Area Hectares", "short_description_en": "Short Description"
}).copy()
for col in ["Name", "Country", "Category", "Short Description"]:
    df[col] = df[col].fillna("").astype(str).str.strip()
df["Category"] = df["Category"].str.title()
df["Area Hectares"] = pd.to_numeric(df["Area Hectares"], errors="coerce")

# UNESCO may represent endangered status as a boolean, a date, or a textual flag.
def truthy_danger(value):
    if pd.isna(value): return False
    if isinstance(value, (bool, np.bool_)): return bool(value)
    if isinstance(value, (int, float, np.number)): return value != 0
    s = str(value).strip().lower()
    return s not in ("", "false", "no", "none", "null", "nan", "0", "not endangered", "not on danger list")

# Prefer explicit danger_list status when present; otherwise fall back to danger.
danger_col = "danger_list" if "danger_list" in df.columns else ("danger" if "danger" in df.columns else None)
if danger_col is None:
    raise ValueError("No danger status field found in UNESCO export.")
df["Danger Status"] = np.where(df[danger_col].map(truthy_danger), "Endangered", "Not endangered")

CATEGORY_COLORS = {"Cultural":"#377eb8", "Natural":"#ff8c00", "Mixed":"#55a868"}
DANGER_COLORS = {"Endangered":"#d62728", "Not endangered":"#7f9cba"}

print("Downloaded from:", SOURCE_URL)
print("Total records:", len(df))
print("Danger status source:", danger_col)
print("Nonmissing area values:", df["Area Hectares"].notna().sum())
print("Positive area values:", (df["Area Hectares"] > 0).sum())
display(df[["Name", "Country", "Category", "Danger Status", "Area Hectares"]].head())

Downloaded from: https://data.unesco.org/api/explore/v2.1/catalog/datasets/whc001/exports/csv?lang=en&timezone=UTC
Total records: 1273
Danger status source: danger_list
Nonmissing area values: 1255
Positive area values: 1170


,Name,Country,Category,Danger Status,Area Hectares
0,The Bony Fish Fossils of the Western Limfjord ...,"[""Denmark""]",Natural,Not endangered,7.000
1,Aalto Works,"[""Finland""]",Cultural,Not endangered,101.131
2,"Beaches of the D-Day Landings, Normandy, 1944","[""France""]",Cultural,Not endangered,4228.680
3,Ancient Buddhist Site of Sarnath,"[""India""]",Cultural,Not endangered,8.050
4,Alamūt Castle and Related Fortifications,"[""Iran (Islamic Republic of)""]",Cultural,Not endangered,672.290


## 1 — NumberOfSitesPerCategory
Number of World Heritage sites in each UNESCO category.

In [2]:
category_counts = df.groupby("Category", as_index=False).size().rename(columns={"size":"Sites"})
fig = px.bar(category_counts, x="Category", y="Sites", color="Category", text="Sites", color_discrete_map=CATEGORY_COLORS, title="Number of UNESCO World Heritage Sites by Category")
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, yaxis_title="Number of Sites")
fig.show()

## 2 — StackedChartOfEndangeredSites
Stacked bars compare endangered and non-endangered sites within each category.

In [3]:
danger_summary = (df.groupby(["Category", "Danger Status"], as_index=False).size().rename(columns={"size":"Sites"}))
fig = px.bar(danger_summary, x="Category", y="Sites", color="Danger Status", text="Sites", barmode="stack", color_discrete_map=DANGER_COLORS, title="World Heritage Sites by Category and Endangered Status — Stacked")
fig.update_layout(yaxis_title="Number of Sites", legend_title_text="Danger List")
fig.show()

## 3 — SideBySideOfEndangeredSites
Side-by-side version of the same comparison.

In [4]:
fig = px.bar(danger_summary, x="Category", y="Sites", color="Danger Status", text="Sites", barmode="group", color_discrete_map=DANGER_COLORS, title="World Heritage Sites by Category and Endangered Status — Side by Side")
fig.update_layout(yaxis_title="Number of Sites", legend_title_text="Danger List")
fig.show()

## 4 — NumberOfHeritageSitesPerCountry
Top countries by number of listed sites. Multi-country listings are kept as provided in `states_names` rather than being counted separately for each country.

In [5]:
TOP_N_COUNTRIES = 20
country_counts = (df.loc[df["Country"].ne("")].groupby("Country", as_index=False).size().rename(columns={"size":"Sites"}).nlargest(TOP_N_COUNTRIES, "Sites").sort_values("Sites"))
fig = px.bar(country_counts, x="Sites", y="Country", orientation="h", text="Sites", title=f"Top {TOP_N_COUNTRIES} Country Labels by Number of World Heritage Sites")
fig.update_layout(height=650, yaxis_title="", xaxis_title="Number of Sites")
fig.show()

## 5 — HeritageAreaPerCountry
Sum of `area_hectares` grouped by UNESCO country label. Missing areas are ignored. For transboundary sites, the full listed area remains associated with the original combined country label.

In [6]:
TOP_N_AREA = 20
area_by_country = (df.loc[df["Country"].ne("")].groupby("Country", as_index=False)["Area Hectares"].sum(min_count=1).dropna().nlargest(TOP_N_AREA, "Area Hectares").sort_values("Area Hectares"))
fig = px.bar(area_by_country, x="Area Hectares", y="Country", orientation="h", title=f"Top {TOP_N_AREA} Country Labels by Total Heritage Area", labels={"Area Hectares":"Total Area (hectares)"}, hover_data={"Area Hectares":":,.0f"})
fig.update_layout(height=650, yaxis_title="")
fig.show()

## 6 — Heritage Sites in France (logarithmic area)
French UNESCO listings, sorted largest to smallest, blue Cultural / orange Natural, with full description on hover. Logarithmic axes require strictly positive area values; excluded values are reported below.

In [7]:
france_all = df[df["Country"].str.contains(r"\bFrance\b", case=False, na=False, regex=True)].copy()
france = france_all[(france_all["Area Hectares"] > 0) & france_all["Category"].isin(["Cultural", "Natural"])].copy()
france = france.sort_values("Area Hectares", ascending=False)
print("France-associated sites:", len(france_all))
print("Plotted positive-area Cultural/Natural sites:", len(france))
print("Excluded (zero/missing/negative area or Mixed):", len(france_all)-len(france))
if france.empty:
    print("No France sites with positive area in this export.")
else:
    fig = px.bar(france, x="Area Hectares", y="Name", color="Category", orientation="h", log_x=True,
                 color_discrete_map=CATEGORY_COLORS,
                 hover_data={"Country":True, "Area Hectares":":,.2f", "Short Description":True, "Name":False},
                 title="UNESCO World Heritage Sites in France — Area (Log Scale)")
    fig.update_layout(height=max(650, len(france)*24), yaxis={"categoryorder":"array", "categoryarray":france["Name"].iloc[::-1].tolist()}, xaxis_title="Area (hectares, logarithmic scale)", yaxis_title="", legend_title_text="Category")
    fig.show()

France-associated sites: 56
Plotted positive-area Cultural/Natural sites: 53
Excluded (zero/missing/negative area or Mixed): 3
